## ETL Pipeline Implementation
This notebook contains the full ETL pipeline for extracting earthquake data from the USGS API, transforming it into a clean `pandas` DataFrame, validating the records, and loading the results into a PostgreSQL database. Each subsequent code cell is labeled with a heading for clarity.


In [ ]:
import os
import re
import logging
from datetime import datetime, timezone

import requests
import pandas as pd
import psycopg2
from psycopg2.extras import execute_values
from dotenv import load_dotenv

# ---------------------------
# Logging setup
# ---------------------------
os.makedirs("logs", exist_ok=True)

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s - %(levelname)s - %(name)s - %(message)s",
    handlers=[
        logging.FileHandler("logs/etl.log", encoding="utf-8"),
        logging.StreamHandler()
    ],
)
logger = logging.getLogger("usgs_etl")

# ---------------------------
# Constants
# ---------------------------
USGS_URL = "https://earthquake.usgs.gov/fdsnws/event/1/query"

NETWORK_MAP = {
    "ak": "Alaska",
    "hv": "Hawaii",
    "ci": "California",
    "nn": "Nevada",
    "us": "USGS",
    "pr": "Puerto Rico",
    "tx": "Texas",
    "nc": "Northern California",
    "uw": "Pacific Northwest",
    "ok": "Oklahoma"
}

MAG_TYPE_MAP = {
    "ml": "Local Magnitude",
    "md": "Duration Magnitude",
    "mb": "Body Wave Magnitude",
    "mww": "Moment Magnitude"
}

LOCATION_PATTERN = r"(?P<distance>\d+)\s+km\s+(?P<direction>[A-Z]+)\s+of\s+(?P<location>.+)"


# ---------------------------
# Extract
# ---------------------------
def fetch_from_usgs() -> dict:
    params = {
        "format": "geojson",
        "starttime": os.getenv("USGS_STARTTIME"),
        "endtime": os.getenv("USGS_ENDTIME"),
        "minmagnitude": float(os.getenv("USGS_MINMAG", "0")),
        "limit": int(os.getenv("USGS_LIMIT", "100")),
        "orderby": os.getenv("USGS_ORDERBY", "time"),
    }

    # Remove None params (if user didn't set start/end)
    params = {k: v for k, v in params.items() if v is not None and v != ""}

    logger.info(f"Fetching USGS data with params={params}")
    resp = requests.get(USGS_URL, params=params, timeout=30)
    resp.raise_for_status()
    data = resp.json()

    count = data.get("metadata", {}).get("count")
    logger.info(f"Fetched {count} events from USGS")
    return data


# ---------------------------
# Transform
# ---------------------------
def transform_usgs_geojson(data: dict) -> pd.DataFrame:
    features = data.get("features", [])
    rows = []

    for f in features:
        props = f.get("properties", {}) or {}
        geom = f.get("geometry", {}) or {}
        coords = (geom.get("coordinates") or [None, None, None])

        lon = coords[0] if len(coords) > 0 else None
        lat = coords[1] if len(coords) > 1 else None
        depth_km = coords[2] if len(coords) > 2 else None

        rows.append({
            "event_id": f.get("id"),
            "mag": props.get("mag"),
            "place": props.get("place"),
            "time_utc": pd.to_datetime(props.get("time"), unit="ms", utc=True, errors="coerce"),
            "updated_utc": pd.to_datetime(props.get("updated"), unit="ms", utc=True, errors="coerce"),
            "status": props.get("status"),
            "tsunami": props.get("tsunami"),
            "sig": props.get("sig"),
            "net": props.get("net"),
            "code": props.get("code"),
            "mag_type": props.get("magType"),
            "event_type": props.get("type"),
            "title": props.get("title"),
            "longitude": lon,
            "latitude": lat,
            "depth_km": depth_km,
        })

    df = pd.DataFrame(rows)

    # Enforce numeric types
    for col in ["mag", "longitude", "latitude", "depth_km", "sig"]:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce")

    # Drop dup event_id
    if "event_id" in df.columns:
        df = df.drop_duplicates(subset=["event_id"], keep="last")

    # Drop columns with >= 50% missing
    missing_percent = df.isna().mean()
    cols_to_drop = missing_percent[missing_percent >= 0.5].index.tolist()
    if cols_to_drop:
        logger.info(f"Dropping columns with >=50% missing: {cols_to_drop}")
        df = df.drop(columns=cols_to_drop)

    # Rename to your final schema naming
    renamed_columns = {
        "mag": "magnitude",
        "place": "location_description",
        "tsunami": "tsunami_flag",
        "sig": "significance_score",
        "net": "network",
        "code": "event_code",
        "mag_type": "magnitude_type",
        "title": "event_title",
        "status": "event_status",
    }
    df = df.rename(columns=renamed_columns)

    # Drop URLs if present (you did this)
    for c in ["detail_url", "event_url", "detail", "url"]:
        if c in df.columns:
            df = df.drop(columns=[c])

    # Round numeric precision (your specs)
    if "magnitude" in df.columns:
        df["magnitude"] = df["magnitude"].round(1)
    if "depth_km" in df.columns:
        df["depth_km"] = df["depth_km"].round(2)
    if "longitude" in df.columns:
        df["longitude"] = df["longitude"].round(3)
    if "latitude" in df.columns:
        df["latitude"] = df["latitude"].round(3)

    # Parse reference location fields from location_description
    if "location_description" in df.columns:
        extracted = df["location_description"].str.extract(LOCATION_PATTERN)
        df["distance_km_from_reference"] = pd.to_numeric(extracted["distance"], errors="coerce")
        df["direction_from_reference"] = extracted["direction"]
        df["reference_location"] = extracted["location"]

        split = df["reference_location"].str.split(",", n=1, expand=True)
        df["reference_city"] = split[0]
        df["reference_region"] = split[1] if split.shape[1] > 1 else None

    # Trim text columns (fix " California" etc.)
    text_cols = df.select_dtypes(include="object").columns
    for col in text_cols:
        df[col] = (
            df[col]
            .astype("string")
            .str.strip()
            .str.replace(r"\s+", " ", regex=True)
        )

    # Enrichment: network and mag types
    if "network" in df.columns:
        df["network_name"] = df["network"].map(NETWORK_MAP)
    if "magnitude_type" in df.columns:
        df["magnitude_type_full"] = df["magnitude_type"].map(MAG_TYPE_MAP)

    # Ensure tsunami_flag boolean
    if "tsunami_flag" in df.columns:
        # USGS typically uses 0/1, sometimes null
        df["tsunami_flag"] = df["tsunami_flag"].fillna(0).astype(int).astype(bool)

    return df


# ---------------------------
# Validate
# ---------------------------
class DataQualityError(Exception):
    pass


def validate_data(df: pd.DataFrame) -> None:
    # Basic expectations (edit freely)
    required_cols = [
        "event_id", "time_utc", "magnitude", "latitude", "longitude", "depth_km",
        "location_description", "network", "magnitude_type"
    ]
    missing = [c for c in required_cols if c not in df.columns]
    if missing:
        raise DataQualityError(f"Missing required columns: {missing}")

    if df.empty:
        raise DataQualityError("No rows returned from extract/transform.")

    if df["event_id"].isna().any():
        raise DataQualityError("event_id contains null values.")

    if df.duplicated(subset=["event_id"]).any():
        raise DataQualityError("Duplicate event_id found after de-duplication.")

    # Range checks
    if ((df["magnitude"] < -2) | (df["magnitude"] > 10)).any():
        raise DataQualityError("Magnitude out of expected range [-2, 10].")

    if ((df["latitude"] < -90) | (df["latitude"] > 90)).any():
        raise DataQualityError("Latitude out of bounds [-90, 90].")

    if ((df["longitude"] < -180) | (df["longitude"] > 180)).any():
        raise DataQualityError("Longitude out of bounds [-180, 180].")

    if ((df["depth_km"] < -10) | (df["depth_km"] > 900)).any():
        raise DataQualityError("Depth out of expected range [-10, 900].")

    # Timestamp sanity (not too far future)
    now = pd.Timestamp.now(tz="UTC")
    if (df["time_utc"] > (now + pd.Timedelta(days=1))).any():
        raise DataQualityError("time_utc contains future timestamps beyond 1 day.")

    logger.info("Data validation passed.")


# ---------------------------
# Load
# ---------------------------
def load_to_postgres(df: pd.DataFrame) -> int:
    host = os.getenv("PG_HOST", "127.0.0.1")
    port = os.getenv("PG_PORT", "5433")
    dbname = os.getenv("PG_DB", "earthquakes_db")
    user = os.getenv("PG_USER", "etl_user")
    password = os.getenv("PG_PASSWORD", "etl_pass")

    logger.info(f"Connecting to Postgres {host}:{port}/{dbname} as {user}")
    conn = psycopg2.connect(
        host=host, port=port, dbname=dbname, user=user, password=password
    )

    # Create table if not exists (matches your final DF columns)
    create_sql = """
    CREATE TABLE IF NOT EXISTS earthquakes (
      event_id text PRIMARY KEY,
      magnitude double precision,
      location_description text,
      time_utc timestamptz NOT NULL,
      updated_utc timestamptz,
      event_status text,
      tsunami_flag boolean,
      significance_score integer,
      network text,
      event_code text,
      magnitude_type text,
      event_type text,
      event_title text,
      longitude double precision,
      latitude double precision,
      depth_km double precision,
      distance_km_from_reference double precision,
      direction_from_reference text,
      reference_location text,
      reference_city text,
      reference_region text,
      network_name text,
      magnitude_type_full text,
      loaded_at timestamptz NOT NULL DEFAULT now()
    );
    """
    cols = [
        "event_id", "magnitude", "location_description", "time_utc", "updated_utc",
        "event_status", "tsunami_flag", "significance_score", "network", "event_code",
        "magnitude_type", "event_type", "event_title",
        "longitude", "latitude", "depth_km",
        "distance_km_from_reference", "direction_from_reference", "reference_location",
        "reference_city", "reference_region", "network_name", "magnitude_type_full"
    ]

    # Ensure all cols exist in df (add missing as null)
    for c in cols:
        if c not in df.columns:
            df[c] = None

    records = [tuple(x) for x in df[cols].to_numpy()]

    upsert_sql = f"""
    INSERT INTO earthquakes ({",".join(cols)})
    VALUES %s
    ON CONFLICT (event_id) DO UPDATE SET
      magnitude = EXCLUDED.magnitude,
      location_description = EXCLUDED.location_description,
      time_utc = EXCLUDED.time_utc,
      updated_utc = EXCLUDED.updated_utc,
      event_status = EXCLUDED.event_status,
      tsunami_flag = EXCLUDED.tsunami_flag,
      significance_score = EXCLUDED.significance_score,
      network = EXCLUDED.network,
      event_code = EXCLUDED.event_code,
      magnitude_type = EXCLUDED.magnitude_type,
      event_type = EXCLUDED.event_type,
      event_title = EXCLUDED.event_title,
      longitude = EXCLUDED.longitude,
      latitude = EXCLUDED.latitude,
      depth_km = EXCLUDED.depth_km,
      distance_km_from_reference = EXCLUDED.distance_km_from_reference,
      direction_from_reference = EXCLUDED.direction_from_reference,
      reference_location = EXCLUDED.reference_location,
      reference_city = EXCLUDED.reference_city,
      reference_region = EXCLUDED.reference_region,
      network_name = EXCLUDED.network_name,
      magnitude_type_full = EXCLUDED.magnitude_type_full,
      loaded_at = now();
    """

    with conn, conn.cursor() as cur:
        cur.execute(create_sql)
        execute_values(cur, upsert_sql, records, page_size=1000)

    conn.close()
    logger.info(f"Loaded {len(records)} rows into Postgres.")
    return len(records)


# ---------------------------
# Runner (Error handling)
# ---------------------------
def run_etl() -> bool:
    start = datetime.now(timezone.utc)
    logger.info("ETL pipeline started.")

    try:
        data = fetch_from_usgs()
        df = transform_usgs_geojson(data)

        logger.info(f"Transform output: rows={len(df)} cols={len(df.columns)}")

        validate_data(df)

        loaded = load_to_postgres(df)

        end = datetime.now(timezone.utc)
        logger.info(f"ETL pipeline finished successfully. loaded_rows={loaded} duration={(end-start).total_seconds():.2f}s")
        return True

    except Exception as e:
        logger.exception(f"ETL failed: {e}")
        return False


if __name__ == "__main__":
    load_dotenv()

    # Run once (simple)
    ok = run_etl()

    # Optional scheduling (uncomment to run every hour)
    # from apscheduler.schedulers.blocking import BlockingScheduler
    # scheduler = BlockingScheduler()
    # scheduler.add_job(run_etl, "cron", minute=0)  # every hour at :00
    # scheduler.start()

    raise SystemExit(0 if ok else 1)

### Import Libraries & Setup Constants


In [ ]:
import os
import re
import logging
from datetime import datetime, timezone

import requests
import pandas as pd
import psycopg2
from psycopg2.extras import execute_values
from dotenv import load_dotenv

# ---------------------------
# Logging setup
# ---------------------------
os.makedirs("logs", exist_ok=True)

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s - %(levelname)s - %(name)s - %(message)s",
    handlers=[
        logging.FileHandler("logs/etl.log", encoding="utf-8"),
        logging.StreamHandler()
    ],
)
logger = logging.getLogger("usgs_etl")

# ---------------------------
# Constants
# ---------------------------
USGS_URL = "https://earthquake.usgs.gov/fdsnws/event/1/query"

NETWORK_MAP = {
    "ak": "Alaska",
    "hv": "Hawaii",
    "ci": "California",
    "nn": "Nevada",
    "us": "USGS",
    "pr": "Puerto Rico",
    "tx": "Texas",
    "nc": "Northern California",
    "uw": "Pacific Northwest",
    "ok": "Oklahoma"
}

MAG_TYPE_MAP = {
    "ml": "Local Magnitude",
    "md": "Duration Magnitude",
    "mb": "Body Wave Magnitude",
    "mww": "Moment Magnitude"
}

LOCATION_PATTERN = r"(?P<distance>\d+)\s+km\s+(?P<direction>[A-Z]+)\s+of\s+(?P<location>.+)"


# ---------------------------
# Extract
# ---------------------------
def fetch_from_usgs() -> dict:
    params = {
        "format": "geojson",
        "starttime": os.getenv("USGS_STARTTIME"),
        "endtime": os.getenv("USGS_ENDTIME"),
        "minmagnitude": float(os.getenv("USGS_MINMAG", "0")),
        "limit": int(os.getenv("USGS_LIMIT", "100")),
        "orderby": os.getenv("USGS_ORDERBY", "time"),
    }

    # Remove None params (if user didn't set start/end)
    params = {k: v for k, v in params.items() if v is not None and v != ""}

    logger.info(f"Fetching USGS data with params={params}")
    resp = requests.get(USGS_URL, params=params, timeout=30)
    resp.raise_for_status()
    data = resp.json()

    count = data.get("metadata", {}).get("count")
    logger.info(f"Fetched {count} events from USGS")
    return data


# ---------------------------
# Transform
# ---------------------------
def transform_usgs_geojson(data: dict) -> pd.DataFrame:
    features = data.get("features", [])
    rows = []

    for f in features:
        props = f.get("properties", {}) or {}
        geom = f.get("geometry", {}) or {}
        coords = (geom.get("coordinates") or [None, None, None])

        lon = coords[0] if len(coords) > 0 else None
        lat = coords[1] if len(coords) > 1 else None
        depth_km = coords[2] if len(coords) > 2 else None

        rows.append({
            "event_id": f.get("id"),
            "mag": props.get("mag"),
            "place": props.get("place"),
            "time_utc": pd.to_datetime(props.get("time"), unit="ms", utc=True, errors="coerce"),
            "updated_utc": pd.to_datetime(props.get("updated"), unit="ms", utc=True, errors="coerce"),
            "status": props.get("status"),
            "tsunami": props.get("tsunami"),
            "sig": props.get("sig"),
            "net": props.get("net"),
            "code": props.get("code"),
            "mag_type": props.get("magType"),
            "event_type": props.get("type"),
            "title": props.get("title"),
            "longitude": lon,
            "latitude": lat,
            "depth_km": depth_km,
        })

    df = pd.DataFrame(rows)

    # Enforce numeric types
    for col in ["mag", "longitude", "latitude", "depth_km", "sig"]:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce")

    # Drop dup event_id
    if "event_id" in df.columns:
        df = df.drop_duplicates(subset=["event_id"], keep="last")

    # Drop columns with >= 50% missing
    missing_percent = df.isna().mean()
    cols_to_drop = missing_percent[missing_percent >= 0.5].index.tolist()
    if cols_to_drop:
        logger.info(f"Dropping columns with >=50% missing: {cols_to_drop}")
        df = df.drop(columns=cols_to_drop)

    # Rename to your final schema naming
    renamed_columns = {
        "mag": "magnitude",
        "place": "location_description",
        "tsunami": "tsunami_flag",
        "sig": "significance_score",
        "net": "network",
        "code": "event_code",
        "mag_type": "magnitude_type",
        "title": "event_title",
        "status": "event_status",
    }
    df = df.rename(columns=renamed_columns)

    # Drop URLs if present (you did this)
    for c in ["detail_url", "event_url", "detail", "url"]:
        if c in df.columns:
            df = df.drop(columns=[c])

    # Round numeric precision (your specs)
    if "magnitude" in df.columns:
        df["magnitude"] = df["magnitude"].round(1)
    if "depth_km" in df.columns:
        df["depth_km"] = df["depth_km"].round(2)
    if "longitude" in df.columns:
        df["longitude"] = df["longitude"].round(3)
    if "latitude" in df.columns:
        df["latitude"] = df["latitude"].round(3)

    # Parse reference location fields from location_description
    if "location_description" in df.columns:
        extracted = df["location_description"].str.extract(LOCATION_PATTERN)
        df["distance_km_from_reference"] = pd.to_numeric(extracted["distance"], errors="coerce")
        df["direction_from_reference"] = extracted["direction"]
        df["reference_location"] = extracted["location"]

        split = df["reference_location"].str.split(",", n=1, expand=True)
        df["reference_city"] = split[0]
        df["reference_region"] = split[1] if split.shape[1] > 1 else None

    # Trim text columns (fix " California" etc.)
    text_cols = df.select_dtypes(include="object").columns
    for col in text_cols:
        df[col] = (
            df[col]
            .astype("string")
            .str.strip()
            .str.replace(r"\s+", " ", regex=True)
        )

    # Enrichment: network and mag types
    if "network" in df.columns:
        df["network_name"] = df["network"].map(NETWORK_MAP)
    if "magnitude_type" in df.columns:
        df["magnitude_type_full"] = df["magnitude_type"].map(MAG_TYPE_MAP)

    # Ensure tsunami_flag boolean
    if "tsunami_flag" in df.columns:
        # USGS typically uses 0/1, sometimes null
        df["tsunami_flag"] = df["tsunami_flag"].fillna(0).astype(int).astype(bool)

    return df


# ---------------------------
# Validate
# ---------------------------
class DataQualityError(Exception):
    pass


def validate_data(df: pd.DataFrame) -> None:
    # Basic expectations (edit freely)
    required_cols = [
        "event_id", "time_utc", "magnitude", "latitude", "longitude", "depth_km",
        "location_description", "network", "magnitude_type"
    ]
    missing = [c for c in required_cols if c not in df.columns]
    if missing:
        raise DataQualityError(f"Missing required columns: {missing}")

    if df.empty:
        raise DataQualityError("No rows returned from extract/transform.")

    if df["event_id"].isna().any():
        raise DataQualityError("event_id contains null values.")

    if df.duplicated(subset=["event_id"]).any():
        raise DataQualityError("Duplicate event_id found after de-duplication.")

    # Range checks
    if ((df["magnitude"] < -2) | (df["magnitude"] > 10)).any():
        raise DataQualityError("Magnitude out of expected range [-2, 10].")

    if ((df["latitude"] < -90) | (df["latitude"] > 90)).any():
        raise DataQualityError("Latitude out of bounds [-90, 90].")

    if ((df["longitude"] < -180) | (df["longitude"] > 180)).any():
        raise DataQualityError("Longitude out of bounds [-180, 180].")

    if ((df["depth_km"] < -10) | (df["depth_km"] > 900)).any():
        raise DataQualityError("Depth out of expected range [-10, 900].")

    # Timestamp sanity (not too far future)
    now = pd.Timestamp.now(tz="UTC")
    if (df["time_utc"] > (now + pd.Timedelta(days=1))).any():
        raise DataQualityError("time_utc contains future timestamps beyond 1 day.")

    logger.info("Data validation passed.")


# ---------------------------
# Load
# ---------------------------
def load_to_postgres(df: pd.DataFrame) -> int:
    host = os.getenv("PG_HOST", "127.0.0.1")
    port = os.getenv("PG_PORT", "5433")
    dbname = os.getenv("PG_DB", "earthquakes_db")
    user = os.getenv("PG_USER", "etl_user")
    password = os.getenv("PG_PASSWORD", "etl_pass")

    logger.info(f"Connecting to Postgres {host}:{port}/{dbname} as {user}")
    conn = psycopg2.connect(
        host=host, port=port, dbname=dbname, user=user, password=password
    )

    # Create table if not exists (matches your final DF columns)
    create_sql = """
    CREATE TABLE IF NOT EXISTS earthquakes (
      event_id text PRIMARY KEY,
      magnitude double precision,
      location_description text,
      time_utc timestamptz NOT NULL,
      updated_utc timestamptz,
      event_status text,
      tsunami_flag boolean,
      significance_score integer,
      network text,
      event_code text,
      magnitude_type text,
      event_type text,
      event_title text,
      longitude double precision,
      latitude double precision,
      depth_km double precision,
      distance_km_from_reference double precision,
      direction_from_reference text,
      reference_location text,
      reference_city text,
      reference_region text,
      network_name text,
      magnitude_type_full text,
      loaded_at timestamptz NOT NULL DEFAULT now()
    );
    """
    cols = [
        "event_id", "magnitude", "location_description", "time_utc", "updated_utc",
        "event_status", "tsunami_flag", "significance_score", "network", "event_code",
        "magnitude_type", "event_type", "event_title",
        "longitude", "latitude", "depth_km",
        "distance_km_from_reference", "direction_from_reference", "reference_location",
        "reference_city", "reference_region", "network_name", "magnitude_type_full"
    ]

    # Ensure all cols exist in df (add missing as null)
    for c in cols:
        if c not in df.columns:
            df[c] = None

    records = [tuple(x) for x in df[cols].to_numpy()]

    upsert_sql = f"""
    INSERT INTO earthquakes ({",".join(cols)})
    VALUES %s
    ON CONFLICT (event_id) DO UPDATE SET
      magnitude = EXCLUDED.magnitude,
      location_description = EXCLUDED.location_description,
      time_utc = EXCLUDED.time_utc,
      updated_utc = EXCLUDED.updated_utc,
      event_status = EXCLUDED.event_status,
      tsunami_flag = EXCLUDED.tsunami_flag,
      significance_score = EXCLUDED.significance_score,
      network = EXCLUDED.network,
      event_code = EXCLUDED.event_code,
      magnitude_type = EXCLUDED.magnitude_type,
      event_type = EXCLUDED.event_type,
      event_title = EXCLUDED.event_title,
      longitude = EXCLUDED.longitude,
      latitude = EXCLUDED.latitude,
      depth_km = EXCLUDED.depth_km,
      distance_km_from_reference = EXCLUDED.distance_km_from_reference,
      direction_from_reference = EXCLUDED.direction_from_reference,
      reference_location = EXCLUDED.reference_location,
      reference_city = EXCLUDED.reference_city,
      reference_region = EXCLUDED.reference_region,
      network_name = EXCLUDED.network_name,
      magnitude_type_full = EXCLUDED.magnitude_type_full,
      loaded_at = now();
    """

    with conn, conn.cursor() as cur:
        cur.execute(create_sql)
        execute_values(cur, upsert_sql, records, page_size=1000)

    conn.close()
    logger.info(f"Loaded {len(records)} rows into Postgres.")
    return len(records)


# ---------------------------
# Runner (Error handling)
# ---------------------------
def run_etl() -> bool:
    start = datetime.now(timezone.utc)
    logger.info("ETL pipeline started.")

    try:
        data = fetch_from_usgs()
        df = transform_usgs_geojson(data)

        logger.info(f"Transform output: rows={len(df)} cols={len(df.columns)}")

        validate_data(df)

        loaded = load_to_postgres(df)

        end = datetime.now(timezone.utc)
        logger.info(f"ETL pipeline finished successfully. loaded_rows={loaded} duration={(end-start).total_seconds():.2f}s")
        return True

    except Exception as e:
        logger.exception(f"ETL failed: {e}")
        return False


if __name__ == "__main__":
    load_dotenv()

    # Run once (simple)
    ok = run_etl()

    # Optional scheduling (uncomment to run every hour)
    # from apscheduler.schedulers.blocking import BlockingScheduler
    # scheduler = BlockingScheduler()
    # scheduler.add_job(run_etl, "cron", minute=0)  # every hour at :00
    # scheduler.start()

    raise SystemExit(0 if ok else 1)

### Full ETL Pipeline Implementation (Version 2)


In [63]:
import requests

USGS_URL = "https://earthquake.usgs.gov/fdsnws/event/1/query"

params = {
    "format": "geojson",        # required
    "starttime": "2026-02-27",  # YYYY-MM-DD
    "endtime": "2026-02-28",
    "minmagnitude": 2.5,        # optional filters
    # "limit": 20000,           # optional
    # "orderby": "time",        # optional: time, time-asc, magnitude, magnitude-asc
}

response = requests.get(USGS_URL, params=params, timeout=30)
response.raise_for_status()
data = response.json()

print("Count:", data.get("metadata", {}).get("count"))
print("First feature keys:", data["features"][0].keys() if data.get("features") else "No results")

Count: 51
First feature keys: dict_keys(['type', 'properties', 'geometry', 'id'])


### Fetch USGS Earthquake Data with Date Filter


In [64]:
import requests

url = "https://earthquake.usgs.gov/fdsnws/event/1/query"

params = {
    "format": "geojson",   # output format
    "orderby": "time",     # sort by event time (most recent first)
    "limit": 10            # limit number of results
}

response = requests.get(url, params=params, timeout=30)
response.raise_for_status()

data = response.json()

# Print summary
print("Total events fetched:", data.get("metadata", {}).get("count"))

# Print some details for each earthquake
for feature in data.get("features", []):
    props = feature["properties"]
    print(
        f"Time: {props['time']}, "
        f"Magnitude: {props['mag']}, "
        f"Place: {props['place']}"
    )

Total events fetched: 10
Time: 1772282248176, Magnitude: 1.6, Place: 82 km ESE of Denali Park, Alaska
Time: 1772281939600, Magnitude: 1.05, Place: 8 km W of Cobb, CA
Time: 1772281317660, Magnitude: 0.989552955258987, Place: 16 km N of Fillmore, CA
Time: 1772281285917, Magnitude: 1.97, Place: 25 km NNW of Mina, Nevada
Time: 1772281081723, Magnitude: 1.8, Place: 19 km NW of Mina, Nevada
Time: 1772280891770, Magnitude: 1.3, Place: 74 km ESE of Denali Park, Alaska
Time: 1772280767258, Magnitude: 1.3, Place: 52 km NW of Skwentna, Alaska
Time: 1772280680583, Magnitude: 1.1, Place: 41 km SSE of Nelchina, Alaska
Time: 1772279446900, Magnitude: 1.82071828471982, Place: 4 km W of Borrego Springs, CA
Time: 1772278932061, Magnitude: 1.64, Place: 89 km NE of Tonopah, Nevada


### Query Latest Earthquakes & Print Summary


In [65]:
import requests
from pprint import pprint

url = "https://earthquake.usgs.gov/fdsnws/event/1/query"

params = {
    "format": "geojson",
    "limit": 1   # we only need one record to inspect columns
}

response = requests.get(url, params=params, timeout=30)
response.raise_for_status()

data = response.json()

print("\nTop-level keys:")
pprint(list(data.keys()))

print("\nMetadata columns:")
pprint(list(data["metadata"].keys()))

if data["features"]:
    feature = data["features"][0]

    print("\nFeature-level keys:")
    pprint(list(feature.keys()))

    print("\nProperties columns:")
    pprint(list(feature["properties"].keys()))

    print("\nGeometry columns:")
    pprint(list(feature["geometry"].keys()))

    print("\nCoordinate structure:")
    pprint(feature["geometry"]["coordinates"])


Top-level keys:
['type', 'metadata', 'features']

Metadata columns:
['generated', 'url', 'title', 'status', 'api', 'limit', 'offset', 'count']

Feature-level keys:
['type', 'properties', 'geometry', 'id']

Properties columns:
['mag',
 'place',
 'time',
 'updated',
 'tz',
 'url',
 'detail',
 'felt',
 'cdi',
 'mmi',
 'alert',
 'status',
 'tsunami',
 'sig',
 'net',
 'code',
 'ids',
 'sources',
 'types',
 'nst',
 'dmin',
 'rms',
 'gap',
 'magType',
 'type',
 'title']

Geometry columns:
['type', 'coordinates']

Coordinate structure:
[-147.309, 63.546, 5]


### Inspect GeoJSON Structure & Available Fields


In [66]:
import pandas as pd

def transform_usgs_geojson(data: dict) -> pd.DataFrame:
    """
    Transform USGS GeoJSON earthquake response into a flat, ETL-friendly DataFrame.
    """
    features = data.get("features", [])
    rows = []

    for f in features:
        props = f.get("properties", {}) or {}
        geom = f.get("geometry", {}) or {}
        coords = (geom.get("coordinates") or [None, None, None])

        # GeoJSON coordinates = [lon, lat, depth_km]
        lon = coords[0] if len(coords) > 0 else None
        lat = coords[1] if len(coords) > 1 else None
        depth_km = coords[2] if len(coords) > 2 else None

        rows.append({
            "event_id": f.get("id"),
            "mag": props.get("mag"),
            "place": props.get("place"),
            "time_utc": pd.to_datetime(props.get("time"), unit="ms", utc=True, errors="coerce"),
            "updated_utc": pd.to_datetime(props.get("updated"), unit="ms", utc=True, errors="coerce"),
            "tz_offset_min": props.get("tz"),
            "url": props.get("url"),
            "detail": props.get("detail"),
            "status": props.get("status"),
            "tsunami": props.get("tsunami"),
            "alert": props.get("alert"),
            "sig": props.get("sig"),
            "net": props.get("net"),
            "code": props.get("code"),
            "mag_type": props.get("magType"),
            "event_type": props.get("type"),
            "title": props.get("title"),
            "longitude": lon,
            "latitude": lat,
            "depth_km": depth_km,
        })

    df = pd.DataFrame(rows)

    # Optional: enforce numeric types cleanly
    for col in ["mag", "longitude", "latitude", "depth_km", "sig"]:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce")

    # Optional: drop duplicates by event_id (safe for incremental loads)
    if "event_id" in df.columns:
        df = df.drop_duplicates(subset=["event_id"], keep="last")

    return df

### Transform USGS GeoJSON to DataFrame


In [67]:
import requests

url = "https://earthquake.usgs.gov/fdsnws/event/1/query"
params = {"format": "geojson", "limit": 100, "orderby": "time"}

resp = requests.get(url, params=params, timeout=30)
resp.raise_for_status()
data = resp.json()

df = transform_usgs_geojson(data)
#print(df.columns.tolist())
df

,event_id,mag,place,time_utc,updated_utc,tz_offset_min,url,detail,status,tsunami,alert,sig,net,code,mag_type,event_type,title,longitude,latitude,depth_km
0,aka2026edlfue,1.600000,"82 km ESE of Denali Park, Alaska",2026-02-28 12:37:28.176000+00:00,2026-02-28 12:38:58.853000+00:00,None,https://earthquake.usgs.gov/earthquakes/eventp...,https://earthquake.usgs.gov/fdsnws/event/1/que...,automatic,0,None,39,ak,a2026edlfue,ml,earthquake,"M 1.6 - 82 km ESE of Denali Park, Alaska",-147.309000,63.546000,5.0000
1,nc75320157,1.050000,"8 km W of Cobb, CA",2026-02-28 12:32:19.600000+00:00,2026-02-28 12:33:57.753000+00:00,None,https://earthquake.usgs.gov/earthquakes/eventp...,https://earthquake.usgs.gov/fdsnws/event/1/que...,automatic,0,None,17,nc,75320157,md,earthquake,"M 1.1 - 8 km W of Cobb, CA",-122.816002,38.835499,2.0500
2,ci41406312,0.989553,"16 km N of Fillmore, CA",2026-02-28 12:21:57.660000+00:00,2026-02-28 12:23:59.792000+00:00,None,https://earthquake.usgs.gov/earthquakes/eventp...,https://earthquake.usgs.gov/fdsnws/event/1/que...,automatic,0,None,15,ci,41406312,ml,earthquake,"M 1.0 - 16 km N of Fillmore, CA",-118.927834,34.539165,12.6300
3,nn00911350,1.970000,"25 km NNW of Mina, Nevada",2026-02-28 12:21:25.917000+00:00,2026-02-28 12:24:27.699000+00:00,None,https://earthquake.usgs.gov/earthquakes/eventp...,https://earthquake.usgs.gov/fdsnws/event/1/que...,automatic,0,None,60,nn,00911350,ml,earthquake,"M 2.0 - 25 km NNW of Mina, Nevada",-118.184800,38.613200,3.4723
4,nn00911348,1.800000,"19 km NW of Mina, Nevada",2026-02-28 12:18:01.723000+00:00,2026-02-28 12:21:06.250000+00:00,None,https://earthquake.usgs.gov/earthquakes/eventp...,https://earthquake.usgs.gov/fdsnws/event/1/que...,automatic,0,None,50,nn,00911348,ml,earthquake,"M 1.8 - 19 km NW of Mina, Nevada",-118.243800,38.527400,0.0324
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
95,aka2026ecsbnj,2.100000,"14 km ESE of Susitna North, Alaska",2026-02-28 02:59:16.833000+00:00,2026-02-28 03:00:52.668000+00:00,None,https://earthquake.usgs.gov/earthquakes/eventp...,https://earthquake.usgs.gov/fdsnws/event/1/que...,automatic,0,None,68,ak,a2026ecsbnj,ml,earthquake,"M 2.1 - 14 km ESE of Susitna North, Alaska",-149.604000,62.099000,46.7000
96,nc75320042,1.090000,"9 km WNW of The Geysers, CA",2026-02-28 02:55:27.150000+00:00,2026-02-28 04:07:21.827000+00:00,None,https://earthquake.usgs.gov/earthquakes/eventp...,https://earthquake.usgs.gov/fdsnws/event/1/que...,automatic,0,None,18,nc,75320042,md,earthquake,"M 1.1 - 9 km WNW of The Geysers, CA",-122.841667,38.819832,2.4700
97,nc75320037,0.220000,"1 km ENE of The Geysers, CA",2026-02-28 02:51:28.040000+00:00,2026-02-28 02:53:02.066000+00:00,None,https://earthquake.usgs.gov/earthquakes/eventp...,https://earthquake.usgs.gov/fdsnws/event/1/que...,automatic,0,None,1,nc,75320037,md,earthquake,"M 0.2 - 1 km ENE of The Geysers, CA",-122.743332,38.782333,-0.7800
98,hv74910626,1.890000,"5 km NE of Pāhala, Hawaii",2026-02-28 02:51:24.860000+00:00,2026-02-28 03:24:26.970000+00:00,None,https://earthquake.usgs.gov/earthquakes/eventp...,https://earthquake.usgs.gov/fdsnws/event/1/que...,reviewed,0,None,55,hv,74910626,ml,earthquake,"M 1.9 - 5 km NE of Pāhala, Hawaii",-155.436167,19.234333,0.3500


### Test Transform & Display Results


In [68]:
import pandas as pd

class DataQualityError(Exception):
    pass

def run_data_quality_checks(df: pd.DataFrame, strict: bool = True) -> dict:
    """
    Runs data quality checks on the transformed USGS earthquakes dataframe.
    Returns a report dict. If strict=True, raises DataQualityError on failures.
    """
    report = {"passed": True, "checks": []}

    def add_check(name: str, passed: bool, details: dict):
        report["checks"].append({"name": name, "passed": passed, "details": details})
        if strict and not passed:
            report["passed"] = False
            raise DataQualityError(f"Data quality check failed: {name} | {details}")
        if not passed:
            report["passed"] = False

    # 1) Required columns exist
    required_cols = [
        "event_id", "time_utc", "mag", "latitude", "longitude", "depth_km",
        "place", "net", "mag_type"
    ]
    missing_cols = [c for c in required_cols if c not in df.columns]
    add_check(
        "required_columns_present",
        passed=(len(missing_cols) == 0),
        details={"missing_columns": missing_cols}
    )

    # 2) event_id not null and unique
    null_event_ids = int(df["event_id"].isna().sum()) if "event_id" in df.columns else len(df)
    dup_event_ids = int(df.duplicated(subset=["event_id"]).sum()) if "event_id" in df.columns else len(df)
    add_check(
        "event_id_non_null",
        passed=(null_event_ids == 0),
        details={"null_event_ids": null_event_ids}
    )
    add_check(
        "event_id_unique",
        passed=(dup_event_ids == 0),
        details={"duplicate_event_ids": dup_event_ids}
    )

    # 3) Timestamp sanity: time_utc present and not in far future
    if "time_utc" in df.columns:
        null_time = int(df["time_utc"].isna().sum())
        add_check("time_utc_non_null", passed=(null_time == 0), details={"null_time_utc": null_time})

        # allow small clock skew; flag anything > 1 day into future
        now = pd.Timestamp.utcnow()
        future_count = int((df["time_utc"] > (now + pd.Timedelta(days=1))).sum())
        add_check(
            "time_utc_not_far_future",
            passed=(future_count == 0),
            details={"future_count": future_count, "threshold": "now + 1 day"}
        )

    # 4) Magnitude range (practical bounds)
    if "mag" in df.columns:
        mag_null = int(df["mag"].isna().sum())
        mag_out_of_range = int(((df["mag"] < -2) | (df["mag"] > 10)).sum())
        add_check("mag_reasonable_range", passed=(mag_out_of_range == 0),
                  details={"null_mag": mag_null, "out_of_range": mag_out_of_range, "range": "[-2, 10]"} )

    # 5) Geo bounds
    if {"latitude", "longitude"}.issubset(df.columns):
        lat_bad = int(((df["latitude"] < -90) | (df["latitude"] > 90)).sum())
        lon_bad = int(((df["longitude"] < -180) | (df["longitude"] > 180)).sum())
        add_check("latitude_in_bounds", passed=(lat_bad == 0), details={"bad_lat_count": lat_bad})
        add_check("longitude_in_bounds", passed=(lon_bad == 0), details={"bad_lon_count": lon_bad})

    # 6) Depth sanity (km): allow earthquakes a bit above sea level (negative small), and down to ~800km+
    if "depth_km" in df.columns:
        depth_bad = int(((df["depth_km"] < -10) | (df["depth_km"] > 900)).sum())
        add_check("depth_km_reasonable_range", passed=(depth_bad == 0),
                  details={"bad_depth_count": depth_bad, "range": "[-10, 900]"} )

    # 7) Place/title basic quality (warn-style; fail only if totally empty)
    if "place" in df.columns:
        empty_place = int(df["place"].fillna("").str.strip().eq("").sum())
        add_check("place_not_empty", passed=(empty_place < len(df)),
                  details={"empty_place_count": empty_place, "total_rows": len(df)})

    # 8) Basic completeness score (not a hard fail unless you want it to be)
    key_cols = ["mag", "time_utc", "latitude", "longitude", "depth_km", "place"]
    present_cols = [c for c in key_cols if c in df.columns]
    completeness = float(1.0 - df[present_cols].isna().mean().mean()) if present_cols and len(df) else 1.0
    add_check("completeness_score", passed=(completeness >= 0.90),
              details={"completeness": round(completeness, 4), "threshold": 0.90, "cols": present_cols})

    return report

### Data Quality Validation Function


In [69]:
df = transform_usgs_geojson(data)

# strict=True => raises error and stops pipeline on failure
dq_report = run_data_quality_checks(df, strict=True)

print("DQ passed:", dq_report["passed"])
for c in dq_report["checks"]:
    print(c["name"], "->", c["passed"])

DQ passed: True
required_columns_present -> True
event_id_non_null -> True
event_id_unique -> True
time_utc_non_null -> True
time_utc_not_far_future -> True
mag_reasonable_range -> True
latitude_in_bounds -> True
longitude_in_bounds -> True
depth_km_reasonable_range -> True
place_not_empty -> True
completeness_score -> True


### Run Data Quality Checks


In [71]:
# Calculate percentage of missing values per column
missing_percent = df.isna().mean()

# Find columns where >= 50% of values are missing
cols_to_drop = missing_percent[missing_percent >= 0.5].index.tolist()

print("Columns with >= 50% missing data:")
print(cols_to_drop)

# Drop those columns
df = df.drop(columns=cols_to_drop)

print("\nRemaining columns:")
print(df.columns.tolist())

Columns with >= 50% missing data:
['tz_offset_min', 'alert']

Remaining columns:
['event_id', 'mag', 'place', 'time_utc', 'updated_utc', 'url', 'detail', 'status', 'tsunami', 'sig', 'net', 'code', 'mag_type', 'event_type', 'title', 'longitude', 'latitude', 'depth_km']


### View Transformed DataFrame


In [73]:
renamed_columns = {
    "mag": "magnitude",
    "place": "location_description",
    "time": "event_time_utc",
    "updated": "last_updated_utc",
    "tz": "timezone_offset_minutes",
    "url": "event_url",
    "detail": "detail_url",
    "felt": "felt_reports_count",
    "cdi": "community_intensity",
    "mmi": "maximum_intensity",
    "alert": "alert_level",
    "status": "event_status",
    "tsunami": "tsunami_flag",
    "sig": "significance_score",
    "net": "network",
    "code": "event_code",
    "ids": "all_event_ids",
    "sources": "data_sources",
    "types": "event_types",
    "nst": "number_of_stations",
    "dmin": "distance_to_nearest_station",
    "rms": "root_mean_square_error",
    "gap": "azimuthal_gap",
    "mag_type": "magnitude_type",
    "type": "event_type",
    "title": "event_title"
}

df = df.rename(columns=renamed_columns)

### Analyze Missing Values Per Column


In [75]:
df = df.drop(columns=['detail_url','event_url'])

### Drop Columns with 50%+ Missing Data


In [76]:
df["magnitude"] = df["magnitude"].round(1)
df["depth_km"] = df["depth_km"].round(2)
df["longitude"] = df["longitude"].round(3)
df["latitude"] = df["latitude"].round(3)

# Verify results
print("\nUpdated dtypes:")
print(df.dtypes)

print("\nSample values:")
print(df[["magnitude", "depth_km", "longitude", "latitude"]].head())


Updated dtypes:
event_id                             object
magnitude                           float64
location_description                 object
time_utc                datetime64[ns, UTC]
updated_utc             datetime64[ns, UTC]
event_status                         object
tsunami_flag                          int64
significance_score                    int64
network                              object
event_code                           object
magnitude_type                       object
event_type                           object
event_title                          object
longitude                           float64
latitude                            float64
depth_km                            float64
dtype: object

Sample values:
   magnitude  depth_km  longitude  latitude
0        1.6      5.00   -147.309    63.546
1        1.0      2.05   -122.816    38.835
2        1.0     12.63   -118.928    34.539
3        2.0      3.47   -118.185    38.613
4        1.8      0.03   -118

### Rename Columns to Final Schema


In [77]:
df.head()

,event_id,magnitude,location_description,time_utc,updated_utc,event_status,tsunami_flag,significance_score,network,event_code,magnitude_type,event_type,event_title,longitude,latitude,depth_km
0,aka2026edlfue,1.6,"82 km ESE of Denali Park, Alaska",2026-02-28 12:37:28.176000+00:00,2026-02-28 12:38:58.853000+00:00,automatic,0,39,ak,a2026edlfue,ml,earthquake,"M 1.6 - 82 km ESE of Denali Park, Alaska",-147.309,63.546,5.00
1,nc75320157,1.0,"8 km W of Cobb, CA",2026-02-28 12:32:19.600000+00:00,2026-02-28 12:33:57.753000+00:00,automatic,0,17,nc,75320157,md,earthquake,"M 1.1 - 8 km W of Cobb, CA",-122.816,38.835,2.05
2,ci41406312,1.0,"16 km N of Fillmore, CA",2026-02-28 12:21:57.660000+00:00,2026-02-28 12:23:59.792000+00:00,automatic,0,15,ci,41406312,ml,earthquake,"M 1.0 - 16 km N of Fillmore, CA",-118.928,34.539,12.63
3,nn00911350,2.0,"25 km NNW of Mina, Nevada",2026-02-28 12:21:25.917000+00:00,2026-02-28 12:24:27.699000+00:00,automatic,0,60,nn,00911350,ml,earthquake,"M 2.0 - 25 km NNW of Mina, Nevada",-118.185,38.613,3.47
4,nn00911348,1.8,"19 km NW of Mina, Nevada",2026-02-28 12:18:01.723000+00:00,2026-02-28 12:21:06.250000+00:00,automatic,0,50,nn,00911348,ml,earthquake,"M 1.8 - 19 km NW of Mina, Nevada",-118.244,38.527,0.03


### View DataFrame After Renaming


In [78]:
df["magnitude_type"].unique()

array(['ml', 'md', 'mb', 'mww'], dtype=object)

### Remove URL Columns


In [79]:
df["network"].unique()

array(['ak', 'nc', 'ci', 'nn', 'ok', 'us', 'hv', 'pr', 'tx', 'uw'],
      dtype=object)

### Round Numeric Fields to Precision


In [80]:
df['location_description']

0       82 km ESE of Denali Park, Alaska
1                     8 km W of Cobb, CA
2                16 km N of Fillmore, CA
3              25 km NNW of Mina, Nevada
4               19 km NW of Mina, Nevada
                     ...                
95    14 km ESE of Susitna North, Alaska
96           9 km WNW of The Geysers, CA
97           1 km ENE of The Geysers, CA
98             5 km NE of Pāhala, Hawaii
99            1 km NE of The Geysers, CA
Name: location_description, Length: 100, dtype: object

### Display First Few Rows


In [81]:
data["metadata"]["count"]

100

### Check Unique Magnitude Types


In [82]:
import re

# Extract structured components
pattern = r"(?P<distance>\d+)\s+km\s+(?P<direction>[A-Z]+)\s+of\s+(?P<location>.+)"

extracted = df["location_description"].str.extract(pattern)

# Add extracted columns
df["distance_km_from_reference"] = extracted["distance"].astype(float)
df["direction_from_reference"] = extracted["direction"]
df["reference_location"] = extracted["location"]

# Optional: split location into city + region
df[["reference_city", "reference_region"]] = (
    df["reference_location"]
    .str.split(",", n=1, expand=True)
)

df.head()

,event_id,magnitude,location_description,time_utc,updated_utc,event_status,tsunami_flag,significance_score,network,event_code,...,event_type,event_title,longitude,latitude,depth_km,distance_km_from_reference,direction_from_reference,reference_location,reference_city,reference_region
0,aka2026edlfue,1.6,"82 km ESE of Denali Park, Alaska",2026-02-28 12:37:28.176000+00:00,2026-02-28 12:38:58.853000+00:00,automatic,0,39,ak,a2026edlfue,...,earthquake,"M 1.6 - 82 km ESE of Denali Park, Alaska",-147.309,63.546,5.00,82.0,ESE,"Denali Park, Alaska",Denali Park,Alaska
1,nc75320157,1.0,"8 km W of Cobb, CA",2026-02-28 12:32:19.600000+00:00,2026-02-28 12:33:57.753000+00:00,automatic,0,17,nc,75320157,...,earthquake,"M 1.1 - 8 km W of Cobb, CA",-122.816,38.835,2.05,8.0,W,"Cobb, CA",Cobb,CA
2,ci41406312,1.0,"16 km N of Fillmore, CA",2026-02-28 12:21:57.660000+00:00,2026-02-28 12:23:59.792000+00:00,automatic,0,15,ci,41406312,...,earthquake,"M 1.0 - 16 km N of Fillmore, CA",-118.928,34.539,12.63,16.0,N,"Fillmore, CA",Fillmore,CA
3,nn00911350,2.0,"25 km NNW of Mina, Nevada",2026-02-28 12:21:25.917000+00:00,2026-02-28 12:24:27.699000+00:00,automatic,0,60,nn,00911350,...,earthquake,"M 2.0 - 25 km NNW of Mina, Nevada",-118.185,38.613,3.47,25.0,NNW,"Mina, Nevada",Mina,Nevada
4,nn00911348,1.8,"19 km NW of Mina, Nevada",2026-02-28 12:18:01.723000+00:00,2026-02-28 12:21:06.250000+00:00,automatic,0,50,nn,00911348,...,earthquake,"M 1.8 - 19 km NW of Mina, Nevada",-118.244,38.527,0.03,19.0,NW,"Mina, Nevada",Mina,Nevada


### Check Unique Seismic Networks


In [83]:
network_map = {
    "ak": "Alaska",
    "hv": "Hawaii",
    "ci": "California",
    "nn": "Nevada",
    "us": "USGS",
    "pr": "Puerto Rico",
    "tx": "Texas",
    "nc": "Northern California",
    "uw": "Pacific Northwest",
    "ok": "Oklahoma"
}

df["network_name"] = df["network"].map(network_map)

### Inspect Location Descriptions


In [84]:
mag_type_map = {
    "ml": "Local Magnitude",
    "md": "Duration Magnitude",
    "mb": "Body Wave Magnitude",
    "mww": "Moment Magnitude"
}

df["magnitude_type_full"] = df["magnitude_type"].map(mag_type_map)

### Get Total Record Count


In [86]:
df.columns

Index(['event_id', 'magnitude', 'location_description', 'time_utc',
       'updated_utc', 'event_status', 'tsunami_flag', 'significance_score',
       'network', 'event_code', 'magnitude_type', 'event_type', 'event_title',
       'longitude', 'latitude', 'depth_km', 'distance_km_from_reference',
       'direction_from_reference', 'reference_location', 'reference_city',
       'reference_region', 'network_name', 'magnitude_type_full'],
      dtype='object')

### Parse Reference Location Components


In [57]:
!pip install psycopg2-binary python-dotenv


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


### Enrich with Network Names


In [88]:
import os
import psycopg2
from psycopg2.extras import execute_values
from dotenv import load_dotenv

load_dotenv()

# Make sure types are correct
df["tsunami_flag"] = df["tsunami_flag"].astype(bool)

conn = psycopg2.connect(
    host="127.0.0.1",
    port="5433",  # host-mapped port
    dbname="earthquakes_db",
    user="etl_user",
    password="etl_pass",
)

cols = [
    "event_id", "magnitude", "location_description", "time_utc", "updated_utc",
    "event_status", "tsunami_flag", "significance_score", "network", "event_code",
    "magnitude_type", "event_type", "event_title",
    "longitude", "latitude", "depth_km",
    "distance_km_from_reference", "direction_from_reference", "reference_location",
    "reference_city", "reference_region", "network_name", "magnitude_type_full"
]

records = [tuple(x) for x in df[cols].to_numpy()]

sql = f"""
INSERT INTO earthquakes ({",".join(cols)})
VALUES %s
ON CONFLICT (event_id) DO UPDATE SET
  magnitude = EXCLUDED.magnitude,
  location_description = EXCLUDED.location_description,
  time_utc = EXCLUDED.time_utc,
  updated_utc = EXCLUDED.updated_utc,
  event_status = EXCLUDED.event_status,
  tsunami_flag = EXCLUDED.tsunami_flag,
  significance_score = EXCLUDED.significance_score,
  network = EXCLUDED.network,
  event_code = EXCLUDED.event_code,
  magnitude_type = EXCLUDED.magnitude_type,
  event_type = EXCLUDED.event_type,
  event_title = EXCLUDED.event_title,
  longitude = EXCLUDED.longitude,
  latitude = EXCLUDED.latitude,
  depth_km = EXCLUDED.depth_km,
  distance_km_from_reference = EXCLUDED.distance_km_from_reference,
  direction_from_reference = EXCLUDED.direction_from_reference,
  reference_location = EXCLUDED.reference_location,
  reference_city = EXCLUDED.reference_city,
  reference_region = EXCLUDED.reference_region,
  network_name = EXCLUDED.network_name,
  magnitude_type_full = EXCLUDED.magnitude_type_full,
  loaded_at = now();
"""

with conn, conn.cursor() as cur:
    execute_values(cur, sql, records, page_size=1000)

conn.close()
print(f"✅ Loaded {len(records)} rows into earthquakes")

✅ Loaded 100 rows into earthquakes
